In [1]:
figure_utils_dir <- if (dir.exists("utils")) "utils" else file.path("code", "figures", "utils")
source(file.path(figure_utils_dir, "common_setup.R"))
source(file.path(figure_utils_dir, "plot_helpers.R"))
source(file.path(figure_utils_dir, "module_helpers.R"))
load_figure_libraries(c("ggtree", "paletteer", "readxl"))

Warning message:
“package ‘Seurat’ was built under R version 4.4.3”
Warning message:
“package ‘SeuratObject’ was built under R version 4.4.3”
Warning message:
“package ‘sp’ was built under R version 4.4.2”
Warning message:
“package ‘qs’ was built under R version 4.4.3”
Warning message:
“package ‘ggplot2’ was built under R version 4.4.3”
Warning message:
“package ‘patchwork’ was built under R version 4.4.3”
Warning message:
“package ‘latex2exp’ was built under R version 4.4.3”
Warning message:
“package ‘ComplexHeatmap’ was built under R version 4.4.2”
Warning message:
“package ‘AnnotationDbi’ was built under R version 4.4.2”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.4.2”
Warning message:
“package ‘Biobase’ was built under R version 4.4.2”
Warning message:
“package ‘IRanges’ was built under R version 4.4.2”
Warning message:
“package ‘S4Vectors’ was built under R version 4.4.2”
Warning message:
“package ‘clusterProfiler’ was built under R version 4.4.2”
Warning 

In [2]:
all_sample_ids <- get_all_sample_ids()
height_width_param <- get_height_width_param()

# 4B cell types

In [3]:
plot_M2_M5_F7_F2 <- lapply(all_sample_ids, function(sample){
    figure_height_width <- get_sample_dimensions(sample, height_width_param)
    obj <- load_xenium_sample(sample)
    plot.tmp <- spatialVariablePlot(obj, plot_cts=c("M2", "M5", "F2", "F7"), focal_ct="F7")
    ggsave(plot=plot.tmp,
           filename=glue("/data1/deyk/harry/RA_Xenium/results/figures/manuscript_final_version_figures/figure4/4B_{sample}_cts.png"),
           height=figure_height_width[1],
           width=figure_height_width[2],
           dpi=500,
           limitsize=FALSE,
           bg="white")
})

# 4B fibrinolysis

In [4]:
all_xenium_merged <- load_merged_xenium_samples(sample_ids = all_sample_ids)

Warning message:
“Some cell names are duplicated across objects provided. Renaming to enforce unique cell names.”
Normalizing layer: counts



In [5]:
fibrolysis_genes <- read_xlsx("/data1/deyk/harry/RA_Xenium/data/Ian_gene_list_xenium_experiments/SAMac_fibrolysis.xlsx")
fibrolysis_genes <- fibrolysis_genes$`Fibrinolysis program`
fibrolysis_genes <- fibrolysis_genes[!is.na(fibrolysis_genes)]
all_xenium_merged <- AddModuleScore(all_xenium_merged, name="Fibrolysis", features=list("Fibrolysis"=fibrolysis_genes), ctrl=20, nbin=24)

In [6]:
all_xenium_merged@meta.data %>%
    filter(celltype_broad=="myeloid") -> all_myeloid_plt

In [7]:
all_fibrolysis_module_score <- lapply(all_sample_ids, function(sample_id){
    figure_height_width <- get_sample_dimensions(sample_id, height_width_param)
    spatialValuePlot(all_myeloid_plt %>% filter(sample==sample_id),
                     palette=paletteer_c("pals::coolwarm", n=50),
                     midpoint=0,
                     pt.size=0.4,
                     stroke=0.01,
                     variable="Fibrolysis1",
                     x_column="y",
                     y_column="x",
                     legend_title="Fibrolysis program",
                     title=sample_id) -> module_score.plt
    ggsave(plot=module_score.plt,
           filename=glue("/data1/deyk/harry/RA_Xenium/results/figures/manuscript_final_version_figures/figure4/4B_fibrinolysis_{sample_id}.png"),
           height=figure_height_width[1],
           width=figure_height_width[2],
           dpi=800,
           limitsize=FALSE)
})